<a href="https://colab.research.google.com/github/5ANDAR8H-5INGH/DOCTOR_PRESCRIPTION_ANALYZER/blob/main/Prescription_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
medicine_dict = pd.read_csv('druglist.csv')
medicine_dict.head(20)

,drug
0,a-to-z-woman-cap-15-s-174701
1,ab-phylline-100mg-strip-of-10-capsules-25470
2,ab-phylline-sr-200mg-tablet-25579
3,absolut-woman-cap-22096
4,absolut-3g-cap-32672
5,acitrom-3mg-strip-of-30-tablets-32843
6,acitrom-2mg-strip-of-30-tablets-32844
7,acitrom-4mg-strip-of-30-tablets-32846
8,acrotac-25mg-strip-of-20-capsules-32934
9,actrapid-flexpen-100iu-pre-fileld-pen-of-3ml-s...


In [ ]:
medicine_dict.isnull().sum()

,0
drug,0


In [ ]:
medicine_dict.duplicated().sum()

np.int64(0)

Data Cleaning

In [ ]:
!pip install ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00


In [ ]:
import ftfy
medicine_dict['drug'] = medicine_dict['drug'].apply(lambda x : ftfy.fix_text(x))

In [ ]:
medicine_dict['drug'] = medicine_dict['drug'].apply(lambda x : x.replace('-', ' '))
display(medicine_dict.head())

,drug
0,a to z woman cap 15 s 174701
1,ab phylline 100mg strip of 10 capsules 25470
2,ab phylline sr 200mg tablet 25579
3,absolut woman cap 22096
4,absolut 3g cap 32672


In [ ]:
import re
medicine_dict['drug'] = medicine_dict['drug'].apply(lambda x: re.sub(r'\s+\d+$', '', str(x)))
# This pattern looks for:
# \s+ (whitespace before the number)
# \d+ (one or more digits)
# $   (at the very end of the string)

In [ ]:
medicine_dict['drug'] = medicine_dict['drug'].apply(lambda x: str(x).lower())

In [ ]:
import re

# Define the list of form words and units to remove
# \d+[mgiu]+ matches numbers followed by mg, g, iu, etc.
# The list of words covers cap, capsule, tablet, tab, strip, etc.
# \bsr\b targets the 'sr' (sustained release) variant specifically
patterns_to_remove = [
    r'\d+\s?[mgiu]+',          # Dosages: 100mg, 3mg, 100iu
    r'\bstrip\b',              # Form: strip
    r'\bof\b',                 # Connector: of
    r'\b\d+\b',                # Standalone numbers: 10, 30, 20
    r'\bcap(sules?)?\b',       # Form: cap, capsule, capsules
    r'\btab(lets?)?\b',        # Form: tab, tablet, tablets
    r'\bsr\b',                 # Variant: sr
    r'\b-s\b'                  # Trailing single 's'
]

# Combine into one regex pattern
combined_pattern = '|'.join(patterns_to_remove)

# Apply the cleaning
# 1. Remove the patterns
# 2. Convert to lower (if not already done)
# 3. Strip extra whitespace left behind
medicine_dict['drug'] = (medicine_dict['drug']
                         .str.replace(combined_pattern, '', regex=True)
                         .str.lower()
                         .str.replace(r'\s+', ' ', regex=True) # Collapse multiple spaces
                         .str.strip())                         # Remove leading/trailing spaces


In [ ]:
medicine_dict[medicine_dict['drug'] == 'paracetamol'].index

Index([163902], dtype='int64')

In [ ]:
medicine_dict['drug'] = medicine_dict['drug'].str.replace(r'\b[mds]\b', '', regex=True)

medicine_dict['drug'] = medicine_dict['drug'].str.replace(r'\s+', ' ', regex=True).str.strip()


In [ ]:
# Remove empty
medicine_dict = medicine_dict[medicine_dict['drug'] != ""]

# Remove short tokens
medicine_dict = medicine_dict[medicine_dict['drug'].str.len() > 4]

# Remove common English words
common_words = [
    "after", "before", "daily", "days", "night",
    "morning", "evening", "meals", "meal",
    "solution", "injection", "tablet", "capsule"
]

medicine_dict = medicine_dict[~medicine_dict['drug'].isin(common_words)]

medicine_dict = medicine_dict.drop_duplicates().reset_index(drop=True)

medicine_dict.head()

,drug
0,a to z woman
1,ab phylline
2,absolut woman
3,absolut
4,acitrom


In [ ]:
# df_2 = pd.read_csv("prescription_text.csv")
# print(df_2.head())

In [ ]:
map = {
    r"\bOD\b": "once daily",
    r"\bBD\b": "twice daily",
    r"\bTDS\b": "three times daily",
    r"\bSOS\b": "when needed",
    r"\bHS\b": "at bedtime"
}

In [ ]:
def clean_text(text):
    text = str(text).lower()
    for abbr, full_form in map.items():
        text = re.sub(abbr.lower(), full_form, text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# df_2["clean_text"] = df_2["prescription_text"].apply(clean_text)
# print(df_2.head())

In [ ]:
# df_2.to_csv("prescription_text_cleaned.csv", index=False)

In [ ]:
df = pd.read_csv("prescription_text_cleaned.csv")

In [ ]:
dosage_pattern = r"\b\d+\s?(?:mg|ml|g|tablet|tab|capsule|cap)\b"
frequency_pattern = r"\b(once daily|twice daily|three times daily|when needed)\b"
timing_pattern = r"\b(after meals?|before meals?|before breakfast|after breakfast|at night|at bedtime|during fever)\b"
duration_pattern = r"\b\d+\s?days?\b"

In [ ]:
def extract_information(text):
    info = {}

    info["dosage"] = re.findall(dosage_pattern, text)
    info["frequency"] = re.findall(frequency_pattern, text)
    info["timing"] = re.findall(timing_pattern, text)
    info["duration"] = re.findall(duration_pattern, text)

    return info

In [ ]:
df["extracted_info"] = df["clean_text"].apply(extract_information)

print(df[["clean_text", "extracted_info"]].head())

                                          clean_text  \
0  tab paracetamol 500mg twice daily after meals ...   
1  tab paracetamol 650mg three times daily after ...   
2        tab paracetamol 500mg when needed for fever   
3           tab brufen 400mg twice daily after meals   
4     tab brufen 600mg three times daily after meals   

                                      extracted_info  
0  {'dosage': ['500mg'], 'frequency': ['twice dai...  
1  {'dosage': ['650mg'], 'frequency': ['three tim...  
2  {'dosage': ['500mg'], 'frequency': ['when need...  
3  {'dosage': ['400mg'], 'frequency': ['twice dai...  
4  {'dosage': ['600mg'], 'frequency': ['three tim...  


In [ ]:
medicine_list = medicine_dict["drug"].tolist()

In [ ]:
def extract_medicine(text):
    found = []
    for med in medicine_list:
        pattern = r"\b" + re.escape(med) + r"\b"
        if re.search(pattern, text):
            found.append(med)
    return found

In [ ]:
df['medicine_name'] = df['clean_text'].apply(extract_medicine)

In [ ]:
df.head()

,prescription_text,label,clean_text,extracted_info,medicine_name
0,Tab Paracetamol 500mg BD after meals 3 days,DO,tab paracetamol 500mg twice daily after meals ...,"{'dosage': ['500mg'], 'frequency': ['twice dai...",[paracetamol]
1,Tab Paracetamol 650mg TDS after meals 3 days,DO,tab paracetamol 650mg three times daily after ...,"{'dosage': ['650mg'], 'frequency': ['three tim...",[paracetamol]
2,Tab Paracetamol 500mg SOS for fever,DO,tab paracetamol 500mg when needed for fever,"{'dosage': ['500mg'], 'frequency': ['when need...",[paracetamol]
3,Tab Brufen 400mg BD after meals,DO,tab brufen 400mg twice daily after meals,"{'dosage': ['400mg'], 'frequency': ['twice dai...",[brufen]
4,Tab Brufen 600mg TDS after meals,DO,tab brufen 600mg three times daily after meals,"{'dosage': ['600mg'], 'frequency': ['three tim...",[brufen]


In [ ]:
df['extracted_info'].iloc[0]

{'dosage': ['500mg'],
 'frequency': ['twice daily'],
 'timing': ['after meals'],
 'duration': ['3 days']}

In [ ]:
df['medicine_name'].iloc[0]

['paracetamol']

In [ ]:
X = df["clean_text"]
y = df["label"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=3000
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
y_pred = model.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9
              precision    recall  f1-score   support

          DO       0.92      1.00      0.96        11
        DONT       0.75      1.00      0.86         3
     WARNING       1.00      0.67      0.80         6

    accuracy                           0.90        20
   macro avg       0.89      0.89      0.87        20
weighted avg       0.92      0.90      0.89        20



In [ ]:
#import joblib
#joblib.dump(model, "prescription_model.pkl")
#joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

In [ ]:
def classify_instruction(text):
    cleaned = clean_text(text)
    vector = vectorizer.transform([cleaned])
    prediction = model.predict(vector)
    return prediction[0]

In [ ]:
classify_instruction("Avoid cold drinks")

'DONT'